### classificazione tipologia dei jobs

##### Categorie ["build", "test", "deploy_release", "lint_security", "eval"]
* build -> compila artefatto
* test -> prende artefatto e testa
* deploy_release -> upload dell'artefatto o su gh o external
* lint_security -> (riorganizzazione codice / repository) & (dependency upgrade / vulnerability check)
* eval -> valutazione modello
* no_ops -> (extra / build docs)

In [1]:
import pandas as pd
import random
random.seed(69)

DATASET_PATH = "C:\\dev\\SE4AI-base\\gigawork\\dataset_with_ids.csv"
BASE_GIGAWORK_PATH = "C:\\dev\\SE4AI-base\\gigawork\\all_workflows"

df = pd.read_csv(DATASET_PATH)

In [2]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))
from src.util.guess_file_type import classify_workflow

In [3]:
# carico i file hash univoci

df_unique = df.drop_duplicates(subset=['workflow_global_id'])

file_hash_by_repo = (
    df_unique.groupby('repository')['file_hash']
    .apply(list)
)

In [17]:
import yaml

result = {}

cnt = 0;

# per la replicabilità
file_hash_by_repo = file_hash_by_repo.sort_index()
file_hash_by_repo = file_hash_by_repo.apply(
    lambda lst: sorted([x for x in lst if pd.notna(x)])
)

for repository_name in file_hash_by_repo.index:
    file_hashes = file_hash_by_repo[repository_name]
    repo_results = []
    for file_hash in file_hashes:
        workflow_path = Path(BASE_GIGAWORK_PATH) / repository_name / f"{file_hash}"
        if workflow_path.exists():
            try:
                with open(workflow_path, "r", encoding="utf-8") as f:
                    data = yaml.safe_load(f)

                steve = data.get('jobs', {})

                for name, job in steve.items():
                    repo_results.append({
                        "file_hash": file_hash,
                        "job_name": name,
                        "job": yaml.dump(job, default_flow_style=True),
                        "SUGGESTED_WORKFLOW_TYPE": classify_workflow(steve)['labels'],
                        "workflow_type": "Giuseppe" if cnt < 50 else ("Vito" if cnt < 100 and cnt >= 50 else None),
                    })
                    cnt += 1
            except Exception:
                continue

    result[repository_name] = repo_results

result

{'AntonOsika__gpt-engineer': [{'file_hash': '04e8eb65ebb244bfdce18f482301c4a5cc739cdcad2a9ebcc2859c84b1db70e1',
   'job_name': 'test',
   'job': "{runs-on: ubuntu-latest, steps: [{name: Checkout repository, uses: actions/checkout@v3},\n    {name: 'Set up Python ${{ matrix.python-version }}', uses: actions/setup-python@v4,\n      with: {cache: pip, python-version: '${{ matrix.python-version == ''3.12'' &&\n          ''3.12.3'' || matrix.python-version }}'}}, {name: Check Python Version,\n      run: python --version}, {name: Install dependencies, run: 'python -m pip install\n        --upgrade pip\n\n        pip install tox==4.15.0 poetry\n\n        '}, {env: {OPENAI_API_KEY: '${{ secrets.OPENAI_API_KEY }}'}, name: Run tox,\n      run: tox}], strategy: {matrix: {python-version: ['3.10', '3.11', '3.12']}}}\n",
   'SUGGESTED_WORKFLOW_TYPE': ['unknown'],
   'workflow_type': 'Giuseppe'},
  {'file_hash': '24fbeb2eaafe191cda4d626104815a037e216834f345c0a9302c1d52e1c97f0f',
   'job_name': 'pre-co

In [18]:
rows = [
    {**item, "repository": repo}
    for repo, items in result.items()
    for item in items
]

df_result = pd.DataFrame(rows)
df_result.to_csv("C:\\dev\\SE4AI-base\\gigawork\\extracted_workflow_types.csv", index=False, encoding="utf-8")